# Runs statistics dashboard

Displays meaningful statistics and visualizations when there are **100+ runs** in the database.

Requires: `RAPBOT_USE_DB=1` and MySQL env vars configured.

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd().parent if "notebooks" in str(Path.cwd()) else Path.cwd()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import os
os.environ.setdefault("RAPBOT_USE_DB", "1")

import json
from collections import Counter, defaultdict
from datetime import datetime

import pandas as pd
import matplotlib.pyplot as plt

%matplotlib inline

: 

In [ ]:
from evo_rhyme import db

MIN_RUNS = 1
ENOUGH_RUNS = False

if not db.db_enabled():
    print("Database not enabled. Set RAPBOT_USE_DB=1 and configure MySQL credentials.")
else:
    total = db.count_runs()
    by_status = db.count_runs_by_status()
    ENOUGH_RUNS = total >= MIN_RUNS
    
    if not ENOUGH_RUNS:
        print(f"Not enough runs: {total} (need {MIN_RUNS}+). Run more evolution experiments first.")
    else:
        print(f"✓ {total} runs in DB. Proceeding with statistics.")
        print(f"  completed: {by_status['by_status']['completed']}, failed: {by_status['by_status']['failed']}, running: {by_status['by_status']['running']}")

## 1. Load runs and outcomes

In [ ]:
# Load all runs (up to 2000)
run_list = db.list_runs(limit=2000, offset=0)
run_ids = [r["run_id"] for r in run_list]

from scripts.analyze_control_impact import load_run_outcomes
rows = load_run_outcomes(run_ids, aggregation_mode="best", top_k=5)

print(f"Loaded {len(run_list)} runs, {len(rows)} with outcomes")

## 2. Summary statistics

In [ ]:
by_status = db.count_runs_by_status()
total = by_status["total"]
status_counts = by_status["by_status"]

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Status pie chart
labels = [k for k, v in status_counts.items() if v > 0]
sizes = [status_counts[k] for k in labels]
colors = {"completed": "#2ecc71", "failed": "#e74c3c", "running": "#3498db"}
ax0 = axes[0]
ax0.pie(sizes, labels=labels, autopct="%1.1f%%", colors=[colors.get(l, "gray") for l in labels], startangle=90)
ax0.set_title("Runs by status")

# Fitness distribution (completed runs with outcomes)
ax1 = axes[1]
fitnesses = [r.get("fitness") or 0 for r in rows if r.get("fitness") is not None]
if fitnesses:
    ax1.hist(fitnesses, bins=30, color="steelblue", edgecolor="white", alpha=0.8)
    ax1.axvline(sum(fitnesses) / len(fitnesses), color="red", linestyle="--", label=f"mean={sum(fitnesses)/len(fitnesses):.3f}")
    ax1.set_xlabel("Best fitness per run")
    ax1.set_ylabel("Count")
    ax1.legend()
else:
    ax1.text(0.5, 0.5, "No fitness data", ha="center", va="center", transform=ax1.transAxes)
ax1.set_title("Fitness distribution")

plt.tight_layout()
plt.show()

print(f"Fitness: n={len(fitnesses)}, min={min(fitnesses):.4f}, max={max(fitnesses):.4f}, mean={sum(fitnesses)/len(fitnesses):.4f}" if fitnesses else "No fitness data.")

## 3. Runs over time

In [ ]:
run_df = pd.DataFrame(run_list)
if "created_at" in run_df.columns:
    run_df["date"] = pd.to_datetime(run_df["created_at"]).dt.date
    daily = run_df.groupby("date").size()
    
    fig, ax = plt.subplots(figsize=(12, 4))
    daily.plot(kind="bar", ax=ax, width=0.8, color="steelblue", alpha=0.8)
    ax.set_xlabel("Date")
    ax.set_ylabel("Runs")
    ax.set_title("Runs per day")
    plt.xticks(rotation=45, ha="right")
    plt.tight_layout()
    plt.show()
else:
    print("No created_at column.")

## 4. Script and theme distribution

In [ ]:
script_counts = Counter(r.get("script_name", "?") for r in run_list)
theme_counts = Counter(r.get("theme_keywords", "") or "(none)" for r in run_list)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

scripts = list(script_counts.keys())[:10]
axes[0].barh(scripts, [script_counts[s] for s in scripts], color="steelblue", alpha=0.8)
axes[0].set_xlabel("Count")
axes[0].set_title("Script distribution (top 10)")

themes = sorted(theme_counts.items(), key=lambda x: -x[1])[:10]
theme_labels = [str(t[0])[:30] + ("…" if len(str(t[0])) > 30 else "") for t in themes]
axes[1].barh(theme_labels, [t[1] for t in themes], color="coral", alpha=0.8)
axes[1].set_xlabel("Count")
axes[1].set_title("Theme distribution (top 10)")

plt.tight_layout()
plt.show()

## 5. Generation and fitness stats (completed runs)

In [ ]:
gen_counts = []
final_best = []
completed_runs = [r for r in run_list if r.get("status") == "completed"]

for r in completed_runs[:500]:  # cap to avoid slow queries
    gens = db.list_generations(r["run_id"])
    gen_counts.append(len(gens))
    if gens:
        final_best.append(gens[-1].get("best_fitness") or 0)

if gen_counts:
    fig, axes = plt.subplots(1, 2, figsize=(10, 4))
    axes[0].hist(gen_counts, bins=25, color="steelblue", edgecolor="white", alpha=0.8)
    axes[0].set_xlabel("Generations per run")
    axes[0].set_ylabel("Count")
    axes[0].set_title(f"Gen count: min={min(gen_counts)}, max={max(gen_counts)}, avg={sum(gen_counts)/len(gen_counts):.1f}")

    if final_best:
        axes[1].hist(final_best, bins=30, color="coral", edgecolor="white", alpha=0.8)
        axes[1].set_xlabel("Final best fitness")
        axes[1].set_ylabel("Count")
        axes[1].set_title(f"Final fitness: mean={sum(final_best)/len(final_best):.4f}")
    plt.tight_layout()
    plt.show()
else:
    print("No generation data for completed runs.")

## 6. Failure analysis (by init, population)

In [ ]:
init_failed = defaultdict(int)
init_completed = defaultdict(int)
pop_failed = defaultdict(int)
pop_completed = defaultdict(int)

for r in run_list:
    cfg = r.get("config_json") or {}
    if isinstance(cfg, str):
        try:
            cfg = json.loads(cfg)
        except Exception:
            cfg = {}
    init = cfg.get("init", cfg.get("control_snapshot", {}).get("init", "?"))
    pop = cfg.get("population", cfg.get("population_size", cfg.get("control_snapshot", {}).get("population", "?")))
    status = (r.get("status") or "?").lower()
    
    if status == "failed":
        init_failed[init] += 1
        pop_failed[pop] += 1
    elif status == "completed":
        init_completed[init] += 1
        pop_completed[pop] += 1

inits = sorted(set(init_failed) | set(init_completed))
pops = sorted(set(pop_failed) | set(pop_completed), key=lambda x: (x == "?", x if isinstance(x, (int, float)) else 0))

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

x = range(len(inits))
axes[0].bar([i - 0.2 for i in x], [init_completed.get(i, 0) for i in inits], 0.4, label="completed", color="#2ecc71", alpha=0.8)
axes[0].bar([i + 0.2 for i in x], [init_failed.get(i, 0) for i in inits], 0.4, label="failed", color="#e74c3c", alpha=0.8)
axes[0].set_xticks(x)
axes[0].set_xticklabels([str(i) for i in inits], rotation=45, ha="right")
axes[0].set_ylabel("Count")
axes[0].set_title("Status by init type")
axes[0].legend()

x2 = range(min(10, len(pops)))
pops_top = pops[:10]
axes[1].bar([i - 0.2 for i in x2], [pop_completed.get(p, 0) for p in pops_top], 0.4, label="completed", color="#2ecc71", alpha=0.8)
axes[1].bar([i + 0.2 for i in x2], [pop_failed.get(p, 0) for p in pops_top], 0.4, label="failed", color="#e74c3c", alpha=0.8)
axes[1].set_xticks(x2)
axes[1].set_xticklabels([str(p) for p in pops_top], rotation=45, ha="right")
axes[1].set_ylabel("Count")
axes[1].set_title("Status by population size (top 10)")
axes[1].legend()

plt.tight_layout()
plt.show()

## 7. Control impact (if controls vary)

In [ ]:
from evo_rhyme.experiment_analysis import analyze_control_impact

report = analyze_control_impact(rows, bootstrap_n=300)
by_control = report.get("by_control") or {}

if by_control:
    # Bar chart: mean fitness by categorical control (first 2 controls)
    for ck, data in list(by_control.items())[:2]:
        ci_by = data.get("ci_by_value") or {}
        if not ci_by:
            continue
        vals = list(ci_by.keys())
        means = [ci_by[v]["mean"] for v in vals]
        lo = [ci_by[v]["ci_low"] for v in vals]
        hi = [ci_by[v]["ci_high"] for v in vals]
        plt.figure(figsize=(8, 4))
        plt.bar(range(len(vals)), means, yerr=[[means[i] - lo[i] for i in range(len(vals))], [hi[i] - means[i] for i in range(len(vals))]], capsize=4)
        plt.xticks(range(len(vals)), [str(v)[:16] for v in vals], rotation=45, ha="right")
        plt.ylabel("fitness (mean ± 95% CI)")
        plt.title(f"Control: {ck}")
        plt.tight_layout()
        plt.show()
else:
    print("No varying controls in config. Control impact analysis skipped.")

## 8. Policy progression

In [ ]:
policy_perf = report.get("policy_performance") or {}

if policy_perf:
    versions = sorted(policy_perf.keys())
    means = [float((policy_perf.get(v) or {}).get("avg_fitness") or 0) for v in versions]
    counts = [int((policy_perf.get(v) or {}).get("n_runs") or 0) for v in versions]
    
    fig, ax = plt.subplots(figsize=(10, 4))
    x = range(len(versions))
    ax.bar([i - 0.2 for i in x], means, 0.4, label="avg fitness", color="steelblue", alpha=0.8)
    ax2 = ax.twinx()
    ax2.bar([i + 0.2 for i in x], counts, 0.4, label="runs", color="coral", alpha=0.5)
    ax.set_xticks(x)
    ax.set_xticklabels([str(v)[:20] for v in versions], rotation=45, ha="right")
    ax.set_ylabel("Avg fitness")
    ax2.set_ylabel("Runs")
    ax.set_title("Policy progression")
    ax.legend(loc="upper left")
    ax2.legend(loc="upper right")
    plt.tight_layout()
    plt.show()
else:
    print("Policy progression skipped: no runs have policy_version in config. "
          "This section applies when runs use learned-policy experiments (e.g. run_continuous with policy_mode: learned).")

## 9. Component score distribution (fitness vector)

In [ ]:
from evo_rhyme.experiment_metrics import FITNESS_VECTOR_KEYS

vec_data = {k: [] for k in FITNESS_VECTOR_KEYS}
for r in rows:
    v = r.get("fitness_vector") or {}
    for k in FITNESS_VECTOR_KEYS:
        if k in v and v[k] is not None:
            vec_data[k].append(float(v[k]))

if any(vec_data.values()):
    fig, ax = plt.subplots(figsize=(10, 4))
    parts = ax.violinplot(
        [vec_data[k] for k in FITNESS_VECTOR_KEYS if vec_data[k]],
        positions=range(len(FITNESS_VECTOR_KEYS)),
        showmeans=True,
    )
    ax.set_xticks(range(len(FITNESS_VECTOR_KEYS)))
    ax.set_xticklabels(FITNESS_VECTOR_KEYS)
    ax.set_ylabel("Score")
    ax.set_title("Fitness vector distribution across runs")
    ax.set_ylim(0, 1.05)
    plt.tight_layout()
    plt.show()
else:
    print("No fitness_vector data.")

## 10. Recent failure rate

## 11. Top runs by fitness (table)

In [ ]:
run_lookup = {x["run_id"]: x for x in run_list}
top_by_fitness = sorted(rows, key=lambda x: x.get("fitness") or 0, reverse=True)[:15]
top_df = pd.DataFrame([
    {
        "run_id": r["run_id"],
        "fitness": round(r.get("fitness") or 0, 4),
        "script": run_lookup.get(r["run_id"], {}).get("script_name", "?"),
        "theme": (run_lookup.get(r["run_id"], {}).get("theme_keywords") or "(none)")[:40],
    }
    for r in top_by_fitness
])
print(top_df.to_string(index=False))

## 12. Rhyme scheme and config distribution

In [ ]:
scheme_counts = Counter()
for r in run_list:
    cfg = r.get("config_json") or {}
    if isinstance(cfg, str):
        try:
            cfg = json.loads(cfg)
        except Exception:
            cfg = {}
    scheme = cfg.get("scheme", cfg.get("rhyme_scheme", cfg.get("control_snapshot", {}).get("scheme", "?")))
    scheme_counts[scheme] += 1

if scheme_counts:
    fig, ax = plt.subplots(figsize=(6, 4))
    ax.bar(list(scheme_counts.keys()), list(scheme_counts.values()), color="teal", alpha=0.8)
    ax.set_xlabel("Rhyme scheme")
    ax.set_ylabel("Count")
    ax.set_title("Runs by rhyme scheme")
    plt.tight_layout()
    plt.show()

## 13. Fitness improvement (gen 0 → final) and diversity/acceptance

In [ ]:
improvements = []
diversities = []
acceptance_rates = []
for r in completed_runs[:500]:
    gens = db.list_generations(r["run_id"])
    if len(gens) >= 2:
        first_best = gens[0].get("best_fitness") or 0
        last_best = gens[-1].get("best_fitness") or 0
        improvements.append(last_best - first_best)
    for g in gens:
        if g.get("diversity") is not None:
            diversities.append(g["diversity"])
        if g.get("acceptance_rate") is not None:
            acceptance_rates.append(g["acceptance_rate"])

n_plots = sum([bool(improvements), bool(diversities), bool(acceptance_rates)])
if n_plots:
    fig, axes = plt.subplots(1, n_plots, figsize=(4 * n_plots, 4))
    if n_plots == 1:
        axes = [axes]
    idx = 0
    if improvements:
        axes[idx].hist(improvements, bins=25, color="green", alpha=0.7, edgecolor="white")
        axes[idx].axvline(0, color="black", linestyle="--")
        axes[idx].set_xlabel("Fitness gain (final - gen 0)")
        axes[idx].set_ylabel("Count")
        axes[idx].set_title(f"Improvement: mean={sum(improvements)/len(improvements):.4f}")
        idx += 1
    if diversities:
        axes[idx].hist(diversities, bins=30, color="purple", alpha=0.7, edgecolor="white")
        axes[idx].set_xlabel("Diversity")
        axes[idx].set_ylabel("Count")
        axes[idx].set_title(f"Diversity: mean={sum(diversities)/len(diversities):.4f}")
        idx += 1
    if acceptance_rates:
        axes[idx].hist(acceptance_rates, bins=30, color="orange", alpha=0.7, edgecolor="white")
        axes[idx].set_xlabel("Acceptance rate")
        axes[idx].set_ylabel("Count")
        axes[idx].set_title(f"Acceptance: mean={sum(acceptance_rates)/len(acceptance_rates):.4f}")
    plt.tight_layout()
    plt.show()

## 14. Run duration and failure reasons

In [ ]:
durations_min = []
for r in run_list:
    created = r.get("created_at")
    updated = r.get("updated_at")
    if created and updated:
        try:
            c = pd.to_datetime(created)
            u = pd.to_datetime(updated)
            delta = (u - c).total_seconds() / 60
            if 0 < delta < 10000:  # filter outliers
                durations_min.append(delta)
        except Exception:
            pass

failure_reasons = Counter()
for r in run_list:
    if (r.get("status") or "").lower() == "failed":
        reason = (r.get("failure_reason") or "unknown")[:80]
        if not reason or reason == "None":
            reason = "unknown"
        failure_reasons[reason] += 1

n_plots = sum([bool(durations_min), bool(failure_reasons)])
if n_plots:
    fig, axes = plt.subplots(1, n_plots, figsize=(6 * n_plots, 4))
    if n_plots == 1:
        axes = [axes]
    idx = 0
    if durations_min:
        axes[idx].hist(durations_min, bins=40, color="slateblue", alpha=0.8)
        axes[idx].set_xlabel("Run duration (minutes)")
        axes[idx].set_ylabel("Count")
        axes[idx].set_title(f"Duration: median={sorted(durations_min)[len(durations_min)//2]:.0f} min")
        idx += 1
    if failure_reasons:
        top_reasons = failure_reasons.most_common(8)
        axes[idx].barh([r[:40] + "…" if len(r) > 40 else r for r, _ in top_reasons], [c for _, c in top_reasons], color="crimson", alpha=0.8)
        axes[idx].set_xlabel("Count")
        axes[idx].set_title("Failure reasons (top 8)")
    plt.tight_layout()
    plt.show()

In [ ]:
recent = run_list[:min(80, len(run_list))]
r_completed = sum(1 for r in recent if (r.get("status") or "").lower() == "completed")
r_failed = sum(1 for r in recent if (r.get("status") or "").lower() == "failed")
r_running = sum(1 for r in recent if (r.get("status") or "").lower() == "running")
total_recent = len(recent)

print(f"Last {total_recent} runs: completed {r_completed} ({100*r_completed/total_recent:.0f}%), failed {r_failed} ({100*r_failed/total_recent:.0f}%), running {r_running} ({100*r_running/total_recent:.0f}%)")